### Extracting resistance marker data from malariagen_data

We want to extract resistance marker data from the [MalariaGEN Vector Genome Observatory](https://malariagen.github.io/vector-data/ag3/ag3.0.html) *An. gambiae* complex phase 3 (Ag3) data resource. This will contain samples from the LLINEUP trial datasets but can easily be modified to include more regions or sample sets.

#### Scope
All unrestricted data from the LLINEUP trial WGS datasets for *An. gambiae* (though *An. funestus* exist too).

#### The data
For each sample, resistance marker genotypes attached to sample metadata (sample set, collection info, location, date of collection, et cetera):
- **SNPs** (*Kdr* L995F/S, *Cyp4j5* L43F, *Coeaed* H232H): number of copies of the resistance allele per sample (0 is wild type, 1 is heterozygote, 2 is homozygote).
- **CNVs** (*Cyp9k1*, *Gstue*, *Cyp6aap* duplications): presence (1) / absence (0) of the duplication.

#### Output
A single .csv vectorscape-sample-resistance-genotypes.csv

#### Note
The triple mutant tagging Dup (Cyp6aa1) seems to give an odd result, so hold off on analysing these until I've marked as fixed.


In [1]:
# Import libs
import malariagen_data
import pandas as pd
import numpy as np
import allel

# init Ag3
ag3 = malariagen_data.Ag3(pre=True, results_cache='../data/ag3_results_cache')

In [2]:
# Let's have a poke around what is available in Ag3 for gambiae in the region

# We can use these if we want to expand our scope
countries = ['Uganda','Kenya','Tanzania','South Sudan', 'Ethiopia','Democratic Republic of the Congo']

sample_sets = ['1288-VO-UG-DONNELLY-VMF00168', '1288-VO-UG-DONNELLY-VMF00219']

sample_query_string = f'sample_set in {sample_sets}' #ex. phenotyped samples from TZ


In [3]:
# List sample sets we are allowed access to
met = ag3.sample_metadata(sample_query=sample_query_string)

# Have a look to see what sample sets these are
met.sample_set.unique() # So these include all of the LLINEUP data

array(['1288-VO-UG-DONNELLY-VMF00168', '1288-VO-UG-DONNELLY-VMF00219'],
      dtype=object)

In [4]:
# Are we allowed to use all of them?
total_samples = len(met)
allowed = np.sum(met.unrestricted_use == True) 
n_countries = len(met.country.unique())

In [5]:
print(f'The total sample count from surrounding countries is: {len(met)} from {n_countries} countries')
print(f'We are allowed to use {allowed}')

The total sample count from surrounding countries is: 1222 from 1 countries
We are allowed to use 1222


In [6]:
# Triple mutant location
# Wait and see

# Build final sample query don't need this as we are just using llineup
#sample_query_string = (f'country in {countries} '
#                        'and sample_set != "1246-VO-TZ-KABULA-VMF00197"'
#                        ' and unrestricted_use == True' #ex. phenotyped samples from TZ
#                      )

# Load metadata
metadata = ag3.sample_metadata(sample_query = sample_query_string)

# Eyeball metadadta
metadata.head()

,sample_id,partner_sample_id,contributor,country,location,year,month,latitude,longitude,sex_call,...,admin1_name,admin1_iso,admin2_name,taxon,cohort_admin1_year,cohort_admin1_month,cohort_admin1_quarter,cohort_admin2_year,cohort_admin2_month,cohort_admin2_quarter
0,VBS10116-4954STDY7089644,UG4A2016A1_96,Mara Lawniczak,Uganda,Busia,2013,1,0.466,34.089,F,...,Eastern Region,UG-E,Busia,gambiae,UG-E_gamb_2013,UG-E_gamb_2013_01,UG-E_gamb_2013_Q1,UG-E_Busia_gamb_2013,UG-E_Busia_gamb_2013_01,UG-E_Busia_gamb_2013_Q1
1,VBS10117-4954STDY7089645,UG4A2016B1_95,Mara Lawniczak,Uganda,Busia,2016,6,0.466,34.089,F,...,Eastern Region,UG-E,Busia,gambiae,UG-E_gamb_2016,UG-E_gamb_2016_06,UG-E_gamb_2016_Q2,UG-E_Busia_gamb_2016,UG-E_Busia_gamb_2016_06,UG-E_Busia_gamb_2016_Q2
2,VBS10118-4954STDY7089646,UG4A2016C1_94,Mara Lawniczak,Uganda,Busia,2016,6,0.466,34.089,F,...,Eastern Region,UG-E,Busia,gambiae,UG-E_gamb_2016,UG-E_gamb_2016_06,UG-E_gamb_2016_Q2,UG-E_Busia_gamb_2016,UG-E_Busia_gamb_2016_06,UG-E_Busia_gamb_2016_Q2
3,VBS10119-4954STDY7089647,UG4A2016D1_93,Mara Lawniczak,Uganda,Busia,2016,6,0.466,34.089,F,...,Eastern Region,UG-E,Busia,gambiae,UG-E_gamb_2016,UG-E_gamb_2016_06,UG-E_gamb_2016_Q2,UG-E_Busia_gamb_2016,UG-E_Busia_gamb_2016_06,UG-E_Busia_gamb_2016_Q2
4,VBS10120-4954STDY7089648,UG4A2016E1_92,Mara Lawniczak,Uganda,Busia,2016,6,0.466,34.089,F,...,Eastern Region,UG-E,Busia,gambiae,UG-E_gamb_2016,UG-E_gamb_2016_06,UG-E_gamb_2016_Q2,UG-E_Busia_gamb_2016,UG-E_Busia_gamb_2016_06,UG-E_Busia_gamb_2016_Q2


#### SNPs
Count copies of the specific resistance allele per sample (0/1/2, `<NA>` = missing genotype). Count the specific allele rather than any alt, as sites can have up to 3 alts. Let's have a poke around the non-*kdr* SNPs first.

In [7]:
# Now for more, get position of L43M
cyp4j5_fqs = ag3.aa_allele_frequencies(transcript= "AGAP006048-RA", sample_query = sample_query_string, cohorts = "admin1_month")
cyp4j5_fqs.query('aa_change == "L43F"')

Load SNP genotypes:   0%|          | 0/596 [00:00<?, ?it/s]

Compute allele frequencies:   0%|          | 0/121 [00:00<?, ?it/s]

Compute SNP effects:   0%|          | 0/1815 [00:00<?, ?it/s]

,,,frq_CD-BC_gamb_2019_12,frq_CD-BC_gamb_2020_01,frq_CD-HK_gamb_2013_01,frq_CD-HK_gamb_2020_01,frq_CD-HU_gamb_2017_01,frq_CD-KC_gamb_2013_01,frq_CD-KC_gamb_2014_01,frq_CD-KC_gamb_2020_08,frq_CD-KN_gamb_2016_01,frq_CD-KN_gamb_2017_01,...,ref_allele,ref_aa,alt_aa,effect,impact,grantham_score,sneath_score,alt_allele,max_af,label
aa_change,contig,position,,,,,,,,,,,,,,,,,,,,,
L43F,2L,25635973,0.5,0.272727,0.677419,0.73913,0.083333,0.25,0.289474,0.194805,0.157143,0.108696,...,G,L,F,NON_SYNONYMOUS_CODING,MODERATE,22.0,22.0,A,1.0,"L43F (2L:25,635,973 G>A)"


In [8]:
# synonyous coeaed snp, what mutational code is this again
coeaed1_fqs = ag3.snp_allele_frequencies(transcript= "AGAP005756-RA", sample_query = sample_query_string, cohorts = "admin1_month")
coeaed1_fqs.query('position == 20288132')

Load SNP genotypes:   0%|          | 0/596 [00:00<?, ?it/s]

Compute allele frequencies:   0%|          | 0/121 [00:00<?, ?it/s]

Compute SNP effects:   0%|          | 0/1829 [00:00<?, ?it/s]

,,,,,pass_gamb_colu_arab,pass_gamb_colu,pass_arab,frq_CD-BC_gamb_2019_12,frq_CD-BC_gamb_2020_01,frq_CD-HK_gamb_2013_01,frq_CD-HK_gamb_2020_01,frq_CD-HU_gamb_2017_01,frq_CD-KC_gamb_2013_01,frq_CD-KC_gamb_2014_01,...,effect,impact,ref_codon,alt_codon,aa_pos,ref_aa,alt_aa,grantham_score,sneath_score,label
contig,position,ref_allele,alt_allele,aa_change,,,,,,,,,,,,,,,,,,,,,
2L,20288132,T,C,H232H,True,True,True,0.525,0.659091,0.370968,0.336957,0.541667,0.46875,0.473684,...,SYNONYMOUS_CODING,LOW,caT,caC,232.0,H,H,NaN,0.0,"2L:20,288,132 T>C (H232H)"


In [9]:
# Init dict of SNPs to extract: name -> (contig, position, ref, alt)
snp_variants = {
    'kdr_L995S':    ('2L', 2422651,  'T', 'C'),
    'kdr_L995F':    ('2L', 2422652,  'A', 'T'),
    'cyp4j5_L43F':  ('2L', 25635973, 'G', 'A'),
    'coeaed_H232H': ('2L', 20288132, 'T', 'C'), # synonymous
}

for name, (contig, pos, ref, alt) in snp_variants.items():
    snp_ds = ag3.snp_calls(region = f'{contig}:{pos}-{pos}', sample_query = sample_query_string)
    snp_alleles = snp_ds.variant_allele.values.astype('U1')
    assert snp_alleles[0, 0] == ref, f'{name}: ref allele mismatch'
    alt_idx = np.flatnonzero(snp_alleles[0] == alt)[0]
    snp_gt = allel.GenotypeArray(snp_ds.call_genotype.values)
    n_alt = snp_gt.to_allele_counts(max_allele = 3)[0, :, alt_idx]
    n_alt = pd.Series(n_alt, index = snp_ds.sample_id.values).astype('Int8').mask(snp_gt.is_missing()[0])
    metadata[name] = metadata.sample_id.map(n_alt) # match on sample_id

metadata[list(snp_variants)].apply(lambda col: col.value_counts(dropna = False)).T

,0,1,2,<NA>
kdr_L995S,2934,475,2300,<NA>
kdr_L995F,4483,581,645,<NA>
cyp4j5_L43F,820,1564,3324,1
coeaed_H232H,1222,2629,1857,1


Sanity check to make sure I haven't messed this up: F and S are in the same codon so can't sit on the same chromosome, meaning no sample should have more than two kdr alleles in total.

In [10]:
assert ((metadata.kdr_L995S + metadata.kdr_L995F) <= 2).all()
pd.crosstab(metadata.kdr_L995S, metadata.kdr_L995F, margins=True)

kdr_L995F,0,1,2,All
kdr_L995S,,,,
0,2015,274,645,2934
1,168,307,0,475
2,2300,0,0,2300
All,4483,581,645,5709


Excellent. The samples with one F and one S are F/S heterozygotes (one allele on each chromosome), which is fine.

#### CNVs
Discordant read calls give presence/absence of a specific duplication, not copy number, so these columns are 0 (absent) / 1 (present). `<NA>` = no call, or the sample has high coverage variance (CNV calls unreliable), or the sample has no CNV data.

In [11]:
# Init dict of CNVs to extract: name -> (contig, variant_id)
cnv_variants = {
    'cyp9k1_dup8':   ('X',  'Cyp9k1_Dup8'),
    'cyp9k1_dup16':  ('X',  'Cyp9k1_Dup16'),
    'gstue_dup2':    ('3R', 'Gstue_Dup2'),
    'gstue_dup3':    ('3R', 'Gstue_Dup3'),
    'cyp6aap_dup1':  ('2R', 'Cyp6aap_Dup1'), #'tagging' cnv for triple mutant
}

cnv_ds = {
    contig: ag3.cnv_discordant_read_calls(contig = contig, sample_query = sample_query_string).set_index(variants = 'variant_id')
    for contig in {contig for contig, _ in cnv_variants.values()}
}

for name, (contig, variant_id) in cnv_variants.items():
    ds = cnv_ds[contig]
    assert variant_id in ds.variants.values, f'{variant_id} not found, options: {list(ds.variants.values)}'
    cnv = ds.sel(variants = variant_id)
    gt = pd.Series(cnv.call_genotype.values, index = cnv.sample_id.values).astype('Int8')
    gt = gt.mask((gt < 0) | cnv.sample_is_high_variance.values)
    metadata[name] = metadata.sample_id.map(gt) # match on sample_id

metadata[list(cnv_variants)].apply(lambda col: col.value_counts(dropna = False)).T

,0,1,<NA>
cyp9k1_dup8,3860,1356,493
cyp9k1_dup16,5086,130,493
gstue_dup2,4923,293,493
gstue_dup3,4814,402,493
cyp6aap_dup1,5210,6,493


In [12]:
# Combine everything and write out
genotype_cols = list(snp_variants) + list(cnv_variants)

# Tidy up first, drop columns we don't want
meta_clean = metadata.loc[:, ~metadata.columns.str.contains('cov|aim|cohort')].drop(columns=['contam_pct','contam_LLR','divergence'])
assert set(genotype_cols) <= set(meta_clean.columns) # make sure the tidy-up didn't drop any

meta_clean.to_csv('vectorscape-sample-resistance-genotypes.csv', index=False)
meta_clean[['sample_id'] + genotype_cols].head()

,sample_id,kdr_L995S,kdr_L995F,cyp4j5_L43F,coeaed_H232H,cyp9k1_dup8,cyp9k1_dup16,gstue_dup2,gstue_dup3,cyp6aap_dup1
0,VBS10116-4954STDY7089644,2,0,1,1,0,0,1,0,0
1,VBS10117-4954STDY7089645,2,0,2,0,0,0,0,0,0
2,VBS10118-4954STDY7089646,2,0,1,2,0,0,0,0,0
3,VBS10119-4954STDY7089647,2,0,2,0,1,0,0,0,0
4,VBS10120-4954STDY7089648,2,0,0,2,<NA>,<NA>,<NA>,<NA>,<NA>


#### Checks
Make sure merging the *kdr* code into the SNP loop hasn't changed anything. Compare against the old outputs: `vectorscape-sample-kdr.csv` (kdr-only, no longer written) and `vectorscape-sample-resistance-genotypes.prev.csv` (a copy of the combined output from before the refactor). Each assert fails loudly if something is off.

In [13]:
# Genotype values should be in range: SNPs 0/1/2, CNVs 0/1, <NA> allowed
snp_vals = meta_clean[list(snp_variants)].melt()['value'].dropna()
cnv_vals = meta_clean[list(cnv_variants)].melt()['value'].dropna()
assert snp_vals.isin([0, 1, 2]).all(), 'SNP value outside 0/1/2'
assert cnv_vals.isin([0, 1]).all(), 'CNV value outside 0/1'

# Summary per marker: missingness and alt allele / duplication frequency
pd.DataFrame({
    'n_called': meta_clean[genotype_cols].notna().sum(),
    'pct_missing': (meta_clean[genotype_cols].isna().mean() * 100).round(1),
    'freq': pd.concat([meta_clean[list(snp_variants)].mean() / 2, meta_clean[list(cnv_variants)].mean()]).round(3),
})

,n_called,pct_missing,freq
kdr_L995S,5709,0.0,0.444
kdr_L995F,5709,0.0,0.164
cyp4j5_L43F,5708,0.0,0.719
coeaed_H232H,5708,0.0,0.556
cyp9k1_dup8,5216,8.6,0.26
cyp9k1_dup16,5216,8.6,0.025
gstue_dup2,5216,8.6,0.056
gstue_dup3,5216,8.6,0.077
cyp6aap_dup1,5216,8.6,0.001


In [14]:
# Independent spot check: pull raw calls for 20 random samples and count the alt allele by letter
rng = np.random.default_rng(1)
check_ids = [str(s) for s in rng.choice(meta_clean.sample_id, 20, replace = False)]
lookup = meta_clean.set_index('sample_id')

for name, (contig, pos, ref, alt) in snp_variants.items():
    ds = ag3.snp_calls(region = f'{contig}:{pos}-{pos}', sample_query = f'sample_id in {check_ids}')
    alleles = ds.variant_allele.values[0].astype('U1')
    for sid, gt in zip(ds.sample_id.values, ds.call_genotype.values[0]):
        expected = pd.NA if (gt < 0).any() else int(sum(alleles[g] == alt for g in gt))
        got = lookup.at[sid, name]
        assert (pd.isna(expected) and pd.isna(got)) or expected == got, f'{name} {sid}: expected {expected}, got {got}'

print(f'OK: {len(snp_variants)} SNPs x {len(check_ids)} samples match raw calls')

OK: 4 SNPs x 20 samples match raw calls


In [ ]:
f